# Modellierung & Evaluation

Verglichen werden klassische Pipelines und generative DeepTabular-Modelle, um die Hypothesen zu Kredit-Scoring-Daten zu testen:
- **H1:** Klassische ML-Modelle erzielen vergleichbare oder bessere Ergebnisse als generative Modelle auf Tabulardaten.
- **H2:** Generative Modelle verursachen einen deutlich höheren Ressourcenbedarf gemessen an Trainingszeit/Rechenaufwand.

## 1️⃣ Vorgehensweise

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from deeptabular.models import MambularClassifier, FTTransformerClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)
from sklearn.base import clone
import time

In [2]:
train_df = pd.read_csv("../data/train/train_fe.csv")
test_df = pd.read_csv("../data/test/test_fe.csv")

print(f"Trainigsdatensatz: {train_df.shape}")
print(f"Testdatensatz: {test_df.shape}")

Trainigsdatensatz: (72878, 60)
Testdatensatz: (50000, 59)


## 2️⃣ Train-Test Split

In [ ]:
X = train_df.drop("Credit_Score", axis=1)
y = train_df["Credit_Score"].astype("category")
label_mapping = {i: cat for i, cat in enumerate(y.cat.categories)}
feature_cols = X.columns.tolist()

X_train, X_test, y_train_cat, y_test_cat = train_test_split(
    X,
    y,
    test_size=0.33,
    random_state=42,
    stratify=y,
)

y_train = y_train_cat.astype(str)
y_test = y_test_cat.astype(str)

y_train_enc = y_train_cat.cat.codes.to_numpy()
y_test_enc = y_test_cat.cat.codes.to_numpy()

print(f"X_Trainigsdatensatz: {X_train.shape}")
print(f"X_Testdatensatz: {X_test.shape}")
print(f"y_Trainigsdatensatz: {y_train.shape}")
print(f"y_Testdatensatz: {y_test.shape}")

X_Trainigsdatensatz: (48828, 59)
X_Testdatensatz: (24050, 59)
y_Trainigsdatensatz: (48828,)
y_Testdatensatz: (24050,)


## 3️⃣ Preprocessing-Pipelines: Numerische & Kategorische Pipelines

- Numerische Features enthalten alle kontinuierlichen und zählenden Engineering-Kennzahlen (z. B. Einkommens-, Delay- und Ratio-Variablen).
- Kategorisch bleiben `Month`, `Occupation` und `Credit_Mix`.
- Verarbeitung: Numerische Werte werden median-imputet und skaliert, kategorische Werte werden mit dem häufigsten Wert imputet und anschließend One-Hot-encodiert.


In [4]:
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

print(f"{len(num_cols)} numerische Merkmale und {len(cat_cols)} kategorielle Merkmale identifiziert.")
print(f"Kategorisch: {cat_cols}")
print(f"Numerisch: {num_cols}")

56 numerische Merkmale und 3 kategorielle Merkmale identifiziert.
Kategorisch: ['Month', 'Occupation', 'Credit_Mix']
Numerisch: ['Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Age', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance', 'Loan_Types_Count', 'Has_Not_Specified_Loan', 'LoanType_Credit-Builder_Loan', 'LoanType_Payday_Loan', 'LoanType_Student_Loan', 'LoanType_Home_Equity_Loan', 'LoanType_Debt_Consolidation_Loan', 'Payment_Spent_Level', 'Payment_Value_Level', 'Is_Payment_Behaviour_Unknown', 'Pays_Min_Amount_Score', 'Is_Min_Payment_Unknown', 'Credit_Mix_Score', 'Credit_History_Months', 'Credit_Age_Gap', 'Credit_History_to_Age', 'Delay_Impact', 'Delay_per_Loan', 'Delayed_Payment_Ratio', 'Debt_To_Income_Ratio', 'EMI_To_Income_Ratio',

In [5]:
numeric_pre = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

In [6]:
categorical_pre = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pre, num_cols),
        ("cat", categorical_pre, cat_cols),
    ],
    remainder="drop",
).set_output(transform="pandas")

preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


Da die DeepTabular Modelle ein anderes Proprocessing benötigen wird für die ein eigenständiger Preprocessor Pipeline erstellt.

In [ ]:
deeptab_preprocessor = clone(preprocessor)

def _clean_columns(df):
    df = df.copy()
    df.columns = df.columns.str.replace("__", "_", regex=False)
    return df

X_train_deeptab = _clean_columns(deeptab_preprocessor.fit_transform(X_train))
X_test_deeptab = _clean_columns(deeptab_preprocessor.transform(X_test))
test_deeptab = _clean_columns(
    deeptab_preprocessor.transform(
        test_df.reindex(columns=feature_cols)
    )
)

## 4️⃣ Modell-Pipelines

- **LR**: Logistic Regression als lineares Baseline-Modell.
- **RF**: RandomForestClassifier als robuster, baumbasierter Klassifikator.
- **HGB**: HistGradientBoostingClassifier als leistungsfähiger Gradient-Boosting-Ansatz.
- **MAM**: MambularClassifier aus DeepTabular für sequentielle Mamba-Blöcke.
- **FTT**: FTTransformerClassifier als attention-basiertes DeepTabular-Modell.


In [9]:
pipe_lr = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", LogisticRegression(
        random_state=42,
    )),
])

pipe_lr

,steps,"[('prep', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [10]:
pipe_rf = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
    )),
])

pipe_rf

,steps,"[('prep', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [11]:
pipe_hgb = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", HistGradientBoostingClassifier(random_state=42)),
])

pipe_hgb

,steps,"[('prep', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
mam_clf = MambularClassifier(
    numerical_preprocessing="standardization",
    categorical_preprocessing="one-hot",
)

In [ ]:
ftt_clf = FTTransformerClassifier(
    numerical_preprocessing="standardization",
    categorical_preprocessing="one-hot",
)

## 5️⃣ Hyperparameter-Tuning

In [ ]:
param_grid_lr = {
    "model__C": [0.1, 1.0],
    "model__class_weight": [None, "balanced"],
    "model__max_iter": [200, 500],
}

param_grid_rf = {
    "model__n_estimators": [200],
    "model__max_depth": [None, 20],
    "model__min_samples_leaf": [1, 4],
    "model__class_weight": [None, "balanced"],
}

param_grid_hgb = {
    "model__learning_rate": [0.03, 0.1],
    "model__max_iter": [100],
    "model__max_leaf_nodes": [31, 127],
    "model__l2_regularization": [0.0, 1.0],
}

Die Epoche wurde auf 1 gesetzt, da in den vorherigen Versuchen die Lauzeit sehr hoch war. Als Nachweis wird hier auf das Notebook `06_DeepTabularHyperparameterTuning` verwiesen.

In [ ]:
fit_params_mam = {
    "max_epochs": 1,
    "rebuild": True,
}

fit_params_ftt = {
    "max_epochs": 1,
    "rebuild": True,
}

In [ ]:
def run_grid_clf_with_time(name, pipe, grid):
    gs = GridSearchCV(
        estimator=pipe,
        param_grid=grid,
        scoring="balanced_accuracy",
        cv=5,
        n_jobs=-1,
        verbose=0,
    )
    
    t0 = time.perf_counter()
    gs.fit(X_train, y_train)
    train_time = time.perf_counter() - t0

    t1 = time.perf_counter()
    y_pred = gs.predict(X_test)
    pred_time = time.perf_counter() - t1

    n_classes = len(np.unique(y_test))
    avg = "binary" if n_classes == 2 else "weighted"

    acc = accuracy_score(y_test, y_pred)
    bacc = balanced_accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average=avg)

    try:
        y_proba = gs.predict_proba(X_test)

        if n_classes == 2:
            auc = roc_auc_score(y_test, y_proba[:, 1])
        else:
            auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="weighted")
            
    except Exception:
        auc = np.nan

    print(f"\n{name}")
    print("-" * 60)
    print(f"Best params:          {gs.best_params_}")
    print(f"CV best bal-acc:      {gs.best_score_:.4f}")
    print(f"Test Accuracy:        {acc:.4f}")
    print(f"Test Balanced Acc.:   {bacc:.4f}")
    print(f"Test F1 ({avg}):      {f1:.4f}")

    if not np.isnan(auc):
        print(f"Test ROC-AUC:         {auc:.4f}")
    else:
        print("Test ROC-AUC:         n/a")

    print(f"Train time (s):       {train_time:.2f}")
    print(f"Predict time (s):     {pred_time:.4f}")

    return {
        "name": name,
        "grid": gs,
        "acc": acc,
        "bacc": bacc,
        "f1": f1,
        "auc": auc,
        "train_time": train_time,
        "pred_time": pred_time,
        "cv_best_bal_acc": gs.best_score_,
    }

In [ ]:
def run_deeptab_clf_with_time(
    name,
    model,
    X_train,
    y_train_enc,
    X_test,
    y_test_enc,
    label_mapping,
    fit_params=None,
):
    if fit_params is None:
        fit_params = {}

    t0 = time.perf_counter()
    model.fit(X_train, y_train_enc, **fit_params)
    train_time = time.perf_counter() - t0

    t1 = time.perf_counter()
    y_pred = model.predict(X_test)
    pred_time = time.perf_counter() - t1

    y_pred_labels = pd.Series(y_pred).map(label_mapping)
    y_test_labels = pd.Series(y_test_enc).map(label_mapping)

    n_classes = len(np.unique(y_train_enc))
    avg = "binary" if n_classes == 2 else "weighted"

    acc = accuracy_score(y_test_labels, y_pred_labels)
    bacc = balanced_accuracy_score(y_test_labels, y_pred_labels)
    f1 = f1_score(y_test_labels, y_pred_labels, average=avg)

    try:
        y_proba = model.predict_proba(X_test)

        if n_classes == 2:
            auc = roc_auc_score(y_test_enc, y_proba[:, 1])
        else:
            auc = roc_auc_score(
                y_test_enc,
                y_proba,
                multi_class="ovr",
                average="weighted",
            )

    except Exception:
        auc = np.nan

    print(f"\n{name}")
    print("-" * 60)
    print("Direkt gesetzte Hyperparameter (ohne Tuning)")
    print(f"Test Accuracy:        {acc:.4f}")
    print(f"Test Balanced Acc.:   {bacc:.4f}")
    print(f"Test F1 ({avg}):      {f1:.4f}")

    if not np.isnan(auc):
        print(f"Test ROC-AUC:         {auc:.4f}")
    else:
        print("Test ROC-AUC:         n/a")

    print(f"Train time (s):       {train_time:.2f}")
    print(f"Predict time (s):     {pred_time:.4f}")

    return {
        "name": name,
        "estimator": model,
        "acc": acc,
        "bacc": bacc,
        "f1": f1,
        "auc": auc,
        "train_time": train_time,
        "pred_time": pred_time,
        "cv_best_bal_acc": np.nan,
    }

In [ ]:
results = []

results.append(run_grid_clf_with_time("LR", pipe_lr, param_grid_lr))
results.append(run_grid_clf_with_time("RF", pipe_rf, param_grid_rf))
results.append(run_grid_clf_with_time("HGB", pipe_hgb, param_grid_hgb))

results.append(
    run_deeptab_clf_with_time(
        "MAM",
        mam_clf,
        X_train_deeptab,
        y_train_enc,
        X_test_deeptab,
        y_test_enc,
        label_mapping,
        fit_params=fit_params_mam,
    )
)

results.append(
    run_deeptab_clf_with_time(
        "FTT",
        ftt_clf,
        X_train_deeptab,
        y_train_enc,
        X_test_deeptab,
        y_test_enc,
        label_mapping,
        fit_params=fit_params_ftt,
    )
)


LR
------------------------------------------------------------
Best params:          {'model__C': 0.1, 'model__class_weight': 'balanced', 'model__max_iter': 200}
CV best bal-acc:      0.7183
Test Accuracy:        0.6739
Test Balanced Acc.:   0.7173
Test F1 (weighted):      0.6788
Test ROC-AUC:         0.8010
Train time (s):       8.07
Predict time (s):     0.0148


/Users/antonioaleksic/Documents/MALE01/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



RF
------------------------------------------------------------
Best params:          {'model__class_weight': 'balanced', 'model__max_depth': None, 'model__min_samples_leaf': 4, 'model__n_estimators': 200}
CV best bal-acc:      0.7785
Test Accuracy:        0.7699
Test Balanced Acc.:   0.7820
Test F1 (weighted):      0.7727
Test ROC-AUC:         0.8888
Train time (s):       105.79
Predict time (s):     0.0836


/Users/antonioaleksic/Documents/MALE01/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores



HGB
------------------------------------------------------------
Best params:          {'model__l2_regularization': 0.0, 'model__learning_rate': 0.1, 'model__max_iter': 100, 'model__max_leaf_nodes': 127}
CV best bal-acc:      0.7602
Test Accuracy:        0.7864
Test Balanced Acc.:   0.7660
Test F1 (weighted):      0.7861
Test ROC-AUC:         0.8976
Train time (s):       35.37
Predict time (s):     0.0780
Numerical Feature: num_Age, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: num_Annual_Income, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: num_Monthly_Inhand_Salary, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: num_Nu

...

/Users/antonioaleksic/Documents/MALE01/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Epoch 0:  18%|█▊        | 54/306 [41:35<3:14:06,  0.02it/s, v_num=0, train_loss_step=1.080]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/Users/antonioaleksic/Documents/MALE01/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 6️⃣ Evaluation

In [ ]:
df_results = pd.DataFrame(results)

family_map = {
    "LR":  "klassisch",
    "RF":  "klassisch",
    "HGB": "klassisch",
    "MAM": "generativ",
    "FTT": "generativ",
}

df_results["family"] = df_results["name"].map(family_map)

print("\nEinzelne Modelle:")
df_results[["name", "family", "bacc", "f1", "auc", "train_time", "pred_time"]]

In [ ]:
print("\nGruppensicht nach Modellfamilie:")

df_family = (
    df_results
    .groupby("family")[["bacc", "f1", "auc", "train_time", "pred_time"]]
    .mean()
    .sort_values("bacc", ascending=False)
)

df_family

In [ ]:
print("\nHypothesenvergleich:")

required_families = {"klassisch", "generativ"}

if required_families.issubset(df_family.index):
    classical = df_family.loc["klassisch"]
    generative = df_family.loc["generativ"]

    h1_supported = classical["bacc"] >= generative["bacc"]
    h1_status = "stützt" if h1_supported else "widerlegt"
    print(
        f"H1 ({h1_status}): bal. acc klassisch={classical['bacc']:.4f} vs. generativ={generative['bacc']:.4f}"
    )

    if classical["train_time"] > 0:
        train_ratio = generative["train_time"] / classical["train_time"]
    else:
        train_ratio = np.nan

    if classical["pred_time"] > 0:
        pred_ratio = generative["pred_time"] / classical["pred_time"]
    else:
        pred_ratio = np.nan

    h2_supported = (
        (not np.isnan(train_ratio) and train_ratio > 1.0)
        or (not np.isnan(pred_ratio) and pred_ratio > 1.0)
    )
    h2_status = "stützt" if h2_supported else "widerlegt"
    train_ratio_txt = f"{train_ratio:.2f}" if not np.isnan(train_ratio) else "nan"
    pred_ratio_txt = f"{pred_ratio:.2f}" if not np.isnan(pred_ratio) else "nan"
    print(
        f"H2 ({h2_status}): train_ratio={train_ratio_txt} | pred_ratio={pred_ratio_txt}"
    )
else:
    print("Nicht alle Modellfamilien vorhanden; H1/H2 können nicht bewertet werden.")

## 7️⃣ Fazit

Die klassischen Pipelines (LR, RF, HGB) liefern nach der Grid-Suche gute Balanced-Accuracy/AUC-Werte bei sehr kurzen Trainings- und Vorhersagezeiten. 

**Damit bestätigt sich Hypothese 1: Für das Kredit-Scoring (tabellarische Daten) reichen „einfache” Machine-Learning-Modelle aus, um eine belastbare Performance zu erzielen, ohne dass komplexe generative Ansätze nötig sind.**

Die Laufzeit der DeepTabular-Modelle (MAM und FTT) mussten auf `max_epochs = 1` limitiert werden, da ein einzelner Fit bereits deutlich länger dauert als die komplette klassische Pipeline (siehe Notebook `06_DeepTabularHyperparameterTuning`). Jeder zusätzliche Hyperparameter- oder Epochen-Schritt würde weitere GPU-Stunden binden.

**Damit bestätigt sich H2: Die generativen DeepTabular-Modelle erzeugen einen unproportional hohen Ressourcenbedarf.**

Für einen erneuten Einsatz der DeepTabular-Modelle wären dedizierte Hardware-Ressourcen oder effizientere Checkpoint-/Frühes-Stopping-Strategien nötig. Solange dies nicht verfügbar ist, bleiben die klassischen Modelle der bevorzugte Weg für ein schnelles und reproduzierbares Kredit-Scoring.